In [1]:
import pandas as pd

In [ ]:
data = pd.read_csv("proteinbase_all_data_28_01_2026.csv")
data

In [ ]:
data

In [ ]:
import json

def unpack_evaluations(df):
    """
    Unpacks all metric fields from the evaluations JSON column into separate columns.
    
    Creates:
    - Suffix-less columns for metrics without a target (e.g. 'esmfold_plddt').
    - Target-specific columns '{metric}_{target}' (e.g. 'kd_human-serum-albumin').
    - Simple '{metric}' columns mapped to the primary/intended designed target.
    """
    # 1. Helper function to extract primary designed target
    def extract_primary_target(evals_json):
        if pd.isna(evals_json) or not isinstance(evals_json, str):
            return None
        try:
            evals = json.loads(evals_json)
            targets = []
            for item in evals:
                if isinstance(item, dict) and "target" in item:
                    t = item["target"]
                    if t and t not in targets:
                        targets.append(t)
            if not targets:
                return None
            control_proteins = {"human-serum-albumin", "fcrn", "hsa"}
            if len(targets) > 1:
                filtered = [t for t in targets if t.lower() not in control_proteins]
                if filtered:
                    targets = filtered
            return targets[0] if len(targets) == 1 else ";".join(targets)
        except:
            return None

    print("Identifying primary targets...")
    df["primary_target"] = df["evaluations"].apply(extract_primary_target)

    print("Unpacking JSON metrics into columns...")
    unpacked_rows = []
    for _, row in df.iterrows():
        row_dict = {}
        evals_json = row["evaluations"]
        prim_target = row["primary_target"]
        
        if isinstance(evals_json, str) and not pd.isna(evals_json):
            try:
                evals = json.loads(evals_json)
                for item in evals:
                    if isinstance(item, dict) and "metric" in item:
                        metric = item["metric"]
                        val = item.get("value")
                        target = item.get("target")
                        
                        # 1. Target-specific column (e.g. kd_nipah-glycoprotein-g)
                        if target:
                            row_dict[f"{metric}_{target}"] = val
                            # 2. General primary metric column (e.g. kd)
                            if target == prim_target:
                                row_dict[metric] = val
                        else:
                            # 3. Suffix-less column for metrics without a target
                            row_dict[metric] = val
            except:
                pass
        unpacked_rows.append(row_dict)
        
    unpacked_df = pd.DataFrame(unpacked_rows, index=df.index)
    res_df = pd.concat([df.drop(columns=["evaluations"]), unpacked_df], axis=1)
    print(f"Success! Expanded dataframe shape to: {res_df.shape}")
    return res_df

# Run the unpacking function
df_unpacked = unpack_evaluations(data)
df_unpacked.head()

In [ ]:
# Print list of some newly created columns
print("=== Unpacked Metric Columns (Subset) ===")
unpacked_cols = [c for c in df_unpacked.columns if c not in data.columns]
print(f"Total new columns created: {len(unpacked_cols)}")
print("\nSample metrics (without targets):")
print([c for c in unpacked_cols if "_" not in c][:15])
print("\nSample target-specific metrics:")
print([c for c in unpacked_cols if "_" in c][:15])